# Notebook 05: Error Type Analysis

**Question:** When models are wrong, are they wrong in the same way? Do false positives and false negatives shift between variants?

**Method:** For each variant, compute the confusion matrix and break errors into false positives (predicted positive, actually negative) and false negatives (predicted negative, actually positive). Compare the FP/FN ratio across variants.

**Why this matters:** In high-stakes classification tasks, false negatives and false positives carry different costs. If quantisation shifts the error composition — even with identical overall accuracy — that is a deployment risk that aggregate metrics would not surface. QuantScope measures whether this shift occurs.

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from quantscope import load_predictions

sns.set_theme(style='whitegrid')

In [ ]:
import os
os.makedirs('../results/charts', exist_ok=True)
print('✅ charts/ folder created')

In [ ]:
data   = load_predictions('../results/all_predictions.json')
labels = np.array(data['labels'])

print(f'Test set: {len(labels)} samples')
print(f'Label distribution: ADR={labels.sum()} | Not ADR={(labels==0).sum()}')

In [ ]:
error_summary = {}

for variant in ['full', 'onnx_fp32', 'onnx_int8']:
    preds = np.array(data[variant]['predictions'])
    cm    = confusion_matrix(labels, preds)

    tn, fp, fn, tp = cm.ravel()
    total_errors   = fp + fn

    error_summary[variant] = {
        'tp'           : int(tp),
        'tn'           : int(tn),
        'fp'           : int(fp),
        'fn'           : int(fn),
        'total_errors' : int(total_errors),
        'fp_pct'       : round(fp / max(total_errors, 1) * 100, 1),
        'fn_pct'       : round(fn / max(total_errors, 1) * 100, 1),
        'precision'    : round(tp / max(tp + fp, 1), 4),
        'recall'       : round(tp / max(tp + fn, 1), 4),
        'f1'           : round(2 * tp / max(2 * tp + fp + fn, 1), 4),
    }

print(f"{'Variant':<14} {'TP':>6} {'TN':>6} {'FP':>6} {'FN':>6} {'FP%':>7} {'FN%':>7} {'F1':>8}")
print('-' * 65)
for variant, s in error_summary.items():
    print(f"{variant:<14} {s['tp']:>6} {s['tn']:>6} {s['fp']:>6} {s['fn']:>6} "
          f"{s['fp_pct']:>6.1f}% {s['fn_pct']:>6.1f}% {s['f1']:>8.4f}")

In [ ]:
for variant in ['full', 'onnx_fp32', 'onnx_int8']:
    preds = np.array(data[variant]['predictions'])
    print(f'\n=== {variant} ===')
    print(classification_report(labels, preds, target_names=['Not ADR', 'ADR']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Error Type Breakdown Across Variants', fontsize=13)

variants  = ['full', 'onnx_fp32', 'onnx_int8']
fp_counts = [error_summary[v]['fp'] for v in variants]
fn_counts = [error_summary[v]['fn'] for v in variants]

x = np.arange(len(variants))
w = 0.35

axes[0].bar(x - w/2, fp_counts, w, label='False Positives', color='#f59e0b')
axes[0].bar(x + w/2, fn_counts, w, label='False Negatives', color='#ef4444')
axes[0].set_xticks(x)
axes[0].set_xticklabels(variants)
axes[0].set_ylabel('Count')
axes[0].set_title('FP vs FN - Absolute Counts')
axes[0].legend()
for i, (fp, fn) in enumerate(zip(fp_counts, fn_counts)):
    axes[0].text(i - w/2, fp + 0.3, str(fp), ha='center', fontsize=9)
    axes[0].text(i + w/2, fn + 0.3, str(fn), ha='center', fontsize=9)

fp_pct = [error_summary[v]['fp_pct'] for v in variants]
fn_pct = [error_summary[v]['fn_pct'] for v in variants]

axes[1].bar(x, fp_pct, label='False Positives (%)', color='#f59e0b')
axes[1].bar(x, fn_pct, bottom=fp_pct, label='False Negatives (%)', color='#ef4444')
axes[1].set_xticks(x)
axes[1].set_xticklabels(variants)
axes[1].set_ylabel('% of total errors')
axes[1].set_title('FP vs FN - Error Composition')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/charts/error_type_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved error_type_breakdown.png')

In [ ]:
with open('../results/error_summary.json', 'w') as f:
    json.dump(error_summary, f, indent=2)

print('Saved error_summary.json')